# Kakao map을 이용한 이수 맛집 검색

In [1]:
# BeautifulSoup + Selenium을 같이 조화롭게 코드를 작성
# 왜?
    # selenium은 브라우저 제어 등 자동화에 아주 특화
    # but Selenium 자체로 HTML 파싱, 데이터 추출에 어느정도 한계가 있다.
    # 그렇기 때문에 각자의 장점을 살려서, 짬뽕

In [2]:
# 라이브러리 import

from selenium import webdriver as wb
from bs4 import BeautifulSoup as bs

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

from tqdm import tqdm

In [3]:
# 브라우저 할당 
driver = wb.Chrome()
driver.get('https://map.kakao.com/')
driver.maximize_window()

In [4]:
# 파란색 팝업 창 기억하는가?
    # 이 부분을 사용자가 클릭해서 없애듯 
    # 컴퓨터에게도 똑같이 시켜줘야한다.
    
click_blue_popup = driver.find_element(By.CSS_SELECTOR,'body > div.coach_layer.coach_layer_type1 > div > div > div > span')
click_blue_popup.click()

In [5]:
# 검색창 지정 (마우스로 올려놓기)

search = driver.find_element(By.CSS_SELECTOR,'#search\.keyword\.query')

# 검색어 입력 ('이수역 맛집')

search.send_keys('이수역 맛집')

# ENTER키 누르고, 데이터가 로드할 수 있는 시간까지 할당해주자

search.send_keys(Keys.ENTER)
time.sleep(3)

### BeautifulSoup 객체화

In [6]:
# 보다 더 빠른 데이터 수집을 위해 -> 추출하는 시간도 이제는 고려

soup = bs(driver.page_source,'lxml')

In [7]:
# 식당이름 (titles)

titles = soup.select('a.link_name')

for i in titles:
    print(i.text)

애플하우스 이수
세녹
스시로로
소담촌 사당직영점
오센
이수회관
호요
전주전집
방배김밥
곱창나라
라화방
호우양꼬치 이수점
훈감동
딥다이브
비엔나커피하우스 이수역점


In [8]:
# 식당 주소 -> addresses

addresses = soup.select('div.addr > p:nth-child(1)')

for i in addresses:
    print(i.text)

서울 동작구 동작대로27다길 29 서광시티뷰 2층 201호
서울 동작구 사당로 304
서울 동작구 동작대로23길 29
서울 동작구 동작대로 43 5층
서울 동작구 동작대로27나길 8
서울 동작구 동작대로27가길 40
서울 동작구 동작대로13길 6-7 1층
서울 동작구 동작대로7길 19
서울 동작구 동작대로27길 59-16
서울 동작구 동작대로27가길 41 1층
서울 동작구 동작대로27길 22
서울 동작구 동작대로27다길 9
서울 동작구 동작대로9길 4 1층
서울 서초구 방배천로18길 36-5 1층
서울 서초구 동작대로 86 1층


In [9]:
# 더보기 버튼 지정 및 클릭

click_more_location = driver.find_element(By.XPATH,'//*[@id="info.search.place.more"]')
click_more_location.click()
time.sleep(2)

In [10]:
# 더보기 버튼을 누른 후에는 , 번호 버튼들이 존재
    # 해당 항목들을 수집

page = driver.find_elements(By.CSS_SELECTOR,'#info\.search\.page > div > a')

### 크롤러 조각하기 

In [11]:
# 이제 본격적으로 크롤러를 작성
    # 레고블럭 조립하듯,
    # 버튼, 요소추출 등 필요한 코드를 미리 작성 -> test
    # 반복문으로 만들 때, 이러한 코드들을 
        # 조각조각 붙여넣어보자
        
# Try Except
    # 예외 처리를 위해 
    # 수집을 완료 했거나, 에러가 발생했을 경우, 
        # 에러를 토해내는 것이 아닌,
        # 성공메시지를 에러메시지가 아닌, '수집완료'라는 메시지가 출력되도록 만들기

In [12]:
# 빈 리스트 2개를 생성

titles_list = []
addresses_list = []

try: # 아래 실행코드들을 시도
    while True: # 조건이 참일 경우
        for i in tqdm(range(6)):
            # 0부터 5까지 총 6번 반복
            
            #만약 순차가 5 미만일 경우 
            if (i < 5):
                # 0부터 4까지일때 참,
                # 5번째 페이지 이상일 때, else로 넘어가 다음버튼을 클릭하게됨
                # BeautifulSoup 객체화
                
                page[i].click()
                time.sleep(2)
                
                soup = bs(driver.page_source,'lxml')
                
                # 식당 이름
                titles = soup.select('a.link_name')
                
                # for문을 돌면서 titles에서 순차적으로 
                for i in titles:
                    # titles_list에 순차적으로 뽑은 요소의 텍스트만 넣어줘
                    titles_list.append(i.text)
                
                # 식당 주소
                addresses = soup.select('div.addr > p:nth-child(1)')
                
                # for문을 돌면서 addresses에서 순차적으로 
                for i in addresses:
                    # adresses_list에 순차적으로 뽑은 요소의 텍스트만 넣어줘
                    addresses_list.append(i.text)
            else:
                # 그렇지 않은 경우는,
                # i가 5보다 큰 경우
                # 5번버튼까지 다 돌고, 다음버튼을 눌러야하는 경우
                
                click_next = driver.find_element(By.XPATH,'//*[@id="info.search.page.next"]')
                click_next.click()
                time.sleep(3)
except :
    print('수집이 완료되었습니다.')
    
driver.quit()

 67%|████████████████████████████████████████████████████████                            | 4/6 [00:08<00:04,  2.18s/it]


수집이 완료되었습니다.


In [13]:
titles_list

['애플하우스 이수',
 '세녹',
 '스시로로',
 '소담촌 사당직영점',
 '오센',
 '이수회관',
 '호요',
 '전주전집',
 '방배김밥',
 '곱창나라',
 '라화방',
 '호우양꼬치 이수점',
 '훈감동',
 '딥다이브',
 '비엔나커피하우스 이수역점',
 '삼육가 사당역1호점',
 '신사펍 이수점',
 '이수통닭 본점',
 '낙성곱창',
 '남미플랜트랩',
 '그릴진',
 '디피스트',
 '리에또',
 '단아한정식',
 '청와옥 사당직영점',
 '올라아보 이수본점',
 '초동집 본점',
 '개화',
 '바른식 시골보쌈&감자옹심이 사당역별관점',
 '부엌쟁이',
 '농부쌈밥',
 '리틀크레이지피자',
 '도너츠윤 방배점',
 '강고집 사당본점',
 '유정우함흥냉면',
 '정작가의막걸리집',
 '초와밥',
 '남궁야',
 '갈비연구소',
 '어사출또 이수역점',
 '요란한부엌',
 '월량',
 '조가네갑오징어 사당점',
 '태양커피',
 '쟝블랑제리',
 '가마솥손두부',
 '페니힐스',
 '스타벅스 이수역점',
 '바다해물포차',
 '도어이스케이프 이수역점',
 '주',
 '정담은',
 '도쿄빙수 이수점',
 '구름떡',
 '이수샤브샤브',
 '속초어시장 방배점',
 '다피타시그니처',
 '이수곱창',
 '규스홈 레스토랑',
 '수향가 사당직영점',
 '방배양곱창',
 '양양메밀막국수',
 '마왕족발 이수역점',
 '37.5 동작이수점',
 '샤브리샤브샤브',
 '원조부안집 이수점',
 '머치커피',
 '태양커피 사당점',
 '김재운초밥사랑 이수점',
 '팬쿡 이수점',
 '파니모들',
 '대구형제막창 방배점',
 '여진족',
 '토방소곱창',
 '맥도날드 이수점',
 '한양왕족발',
 '참오리전문점',
 '남성집',
 '용용선생 이수점',
 '정담이깃든사랑채',
 '헬무트',
 '라무진 이수점',
 '김부삼 사당역점',
 '스타벅스 이수역사거리점',
 '인기명 이수점',
 '보태닉마켓',
 '이태리양조장',
 '남해안활어',
 '싱싱오동도 산아나

In [18]:
len(titles_list),len(addresses_list)

(500, 500)

In [22]:
kakao_data = {'식당 이름' : titles_list, '식당 주소' : addresses_list}
kakao_data

{'식당 이름': ['애플하우스 이수',
  '세녹',
  '스시로로',
  '소담촌 사당직영점',
  '오센',
  '이수회관',
  '호요',
  '전주전집',
  '방배김밥',
  '곱창나라',
  '라화방',
  '호우양꼬치 이수점',
  '훈감동',
  '딥다이브',
  '비엔나커피하우스 이수역점',
  '삼육가 사당역1호점',
  '신사펍 이수점',
  '이수통닭 본점',
  '낙성곱창',
  '남미플랜트랩',
  '그릴진',
  '디피스트',
  '리에또',
  '단아한정식',
  '청와옥 사당직영점',
  '올라아보 이수본점',
  '초동집 본점',
  '개화',
  '바른식 시골보쌈&감자옹심이 사당역별관점',
  '부엌쟁이',
  '농부쌈밥',
  '리틀크레이지피자',
  '도너츠윤 방배점',
  '강고집 사당본점',
  '유정우함흥냉면',
  '정작가의막걸리집',
  '초와밥',
  '남궁야',
  '갈비연구소',
  '어사출또 이수역점',
  '요란한부엌',
  '월량',
  '조가네갑오징어 사당점',
  '태양커피',
  '쟝블랑제리',
  '가마솥손두부',
  '페니힐스',
  '스타벅스 이수역점',
  '바다해물포차',
  '도어이스케이프 이수역점',
  '주',
  '정담은',
  '도쿄빙수 이수점',
  '구름떡',
  '이수샤브샤브',
  '속초어시장 방배점',
  '다피타시그니처',
  '이수곱창',
  '규스홈 레스토랑',
  '수향가 사당직영점',
  '방배양곱창',
  '양양메밀막국수',
  '마왕족발 이수역점',
  '37.5 동작이수점',
  '샤브리샤브샤브',
  '원조부안집 이수점',
  '머치커피',
  '태양커피 사당점',
  '김재운초밥사랑 이수점',
  '팬쿡 이수점',
  '파니모들',
  '대구형제막창 방배점',
  '여진족',
  '토방소곱창',
  '맥도날드 이수점',
  '한양왕족발',
  '참오리전문점',
  '남성집',
  '용용선생 이수점',
  '정담이깃든사랑채',
  '헬무트',
  '라무진

In [23]:
import pandas as pd
kakaos_isu_data = pd.DataFrame(kakao_data)
kakaos_isu_data

,식당 이름,식당 주소
0,애플하우스 이수,서울 동작구 동작대로27다길 29 서광시티뷰 2층 201호
1,세녹,서울 동작구 사당로 304
2,스시로로,서울 동작구 동작대로23길 29
3,소담촌 사당직영점,서울 동작구 동작대로 43 5층
4,오센,서울 동작구 동작대로27나길 8
...,...,...
495,방배동쌀국수,서울 서초구 방배천로8길 34
496,명동참치,서울 동작구 동작대로1길 18 2층
497,한솥도시락 이수역서문여고점,서울 서초구 동광로12가길 21
498,정겨운늑대,서울 서초구 방배중앙로21길 6


In [24]:
# 크롤링하고 혹여 결과가 중복되게 뽑히는 경우가 있다.
# 물론 코드를 보수하는 방법이 좋지만,
    # 우선 DataFrame으로 뽑아보고,
    # 중복값을 제거해주는 방법도 있다.
    
kakaos_isu_data.drop_duplicates()

,식당 이름,식당 주소
0,애플하우스 이수,서울 동작구 동작대로27다길 29 서광시티뷰 2층 201호
1,세녹,서울 동작구 사당로 304
2,스시로로,서울 동작구 동작대로23길 29
3,소담촌 사당직영점,서울 동작구 동작대로 43 5층
4,오센,서울 동작구 동작대로27나길 8
...,...,...
495,방배동쌀국수,서울 서초구 방배천로8길 34
496,명동참치,서울 동작구 동작대로1길 18 2층
497,한솥도시락 이수역서문여고점,서울 서초구 동광로12가길 21
498,정겨운늑대,서울 서초구 방배중앙로21길 6


In [27]:
# 파일을 엑셀로 내보내기
kakaos_isu_data.to_excel('C:/Users/user07/Desktop/isu.xlsx',encoding='euc-kr')

C:\Users\user07\anaconda3\lib\site-packages\pandas\util\_decorators.py:211: FutureWarning: the 'encoding' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'encoding'
  return func(*args, **kwargs)
